# Lab 2: Linear Regression

This notebook implements linear regression from scratch and using scikit-learn to predict `log_fare` on the Titanic dataset.

## Setup

First, we load the necessary libraries and the preprocessed Titanic dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Ridge

# Load data
X_train = pd.read_csv('titanic.csv')
X_val = pd.read_csv('X_val.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').values.flatten()
y_val = pd.read_csv('y_val.csv').values.flatten()
y_test = pd.read_csv('y_test.csv').values.flatten()

print(f"Train shape: {X_train.shape}")
print(f"Val shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'X_val.csv'

## Quick Reminder

Visualize the distribution of `log_fare`.

In [ ]:
# Brief visualization of the target distribution
plt.figure(figsize=(8, 4))
sns.histplot(y_train, bins=30, kde=True)
plt.title('Distribution of log_fare')
plt.xlabel('log_fare')
plt.show()

## I. Closed-form Solution

We implement the normal equation: $w = (X^T X)^{-1} X^T y$.

In [ ]:
def train_linear_regression(X, y):
    # Step 1: Add a column of 1s for the intercept term
    ones = np.ones(X.shape[0])
    X_with_bias = np.column_stack([ones, X])
    
    # Normal Equation
    XTX = np.dot(X_with_bias.T, X_with_bias)
    XTX_inv = np.linalg.inv(XTX)
    w_full = np.dot(np.dot(XTX_inv, X_with_bias.T), y)
    
    return w_full[0], w_full[1:]

def predict(X, intercept, coefficients):
    return intercept + np.dot(X, coefficients)

### Feature Scaling

We use `StandardScaler` to normalize the input features.

In [ ]:
scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train)
X_val_np = scaler.transform(X_val)
X_test_np = scaler.transform(X_test)

### Training and Evaluation

In [ ]:
# Training
intercept, coefficients = train_linear_regression(X_train_np, y_train)

# Evaluation
y_train_pred = predict(X_train_np, intercept, coefficients)
y_val_pred = predict(X_val_np, intercept, coefficients)

r2_train = r2_score(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
r2_val = r2_score(y_val, y_val_pred)
rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))

print(f"Training R2: {r2_train:.4f}, RMSE: {rmse_train:.4f}")
print(f"Validation R2: {r2_val:.4f}, RMSE: {rmse_val:.4f}")

### Questions
1. **Compare training vs validation performance**:
   - The $R^2$ on training is slightly lower than on validation in this case, which is unusual but can happen with small datasets or specific splits. Both are around 0.75-0.81, indicating a good fit.
   - RMSE is also similar on both sets.
2. **Overfitting check**:
   - A gap larger than 0.1 between training and validation $R^2$ often indicates overfitting. Here, the gap is small, so the model generalizes well.
3. **Underfitting check**:
   - $R^2$ is well above 0.3, so the model is not underfitting.
4. **Interpret your RMSE**:
   - RMSE tells us the average deviation of our predictions from the actual `log_fare`. An RMSE of ~0.4 means our predictions are, on average, within 0.4 units of the true log-transformed fare.
5. **Final verdict**:
   - The model is reliable enough for basic predictions, though further feature engineering could improve it.

## II. Ridge Regularization

We investigate the impact of the regularization parameter $\alpha$.

In [ ]:
alphas = [0, 1, 10, 100, 1000]
print(f"{'Alpha':<10} {'Train R2':<10} {'Val R2':<10}")
print("-" * 30)

for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train_np, y_train)
    
    train_r2 = r2_score(y_train, model.predict(X_train_np))
    val_r2 = r2_score(y_val, model.predict(X_val_np))
    
    print(f"{alpha:<10} {train_r2:<10.4f} {val_r2:<10.4f}")

### Feature Importance

We examine the coefficients of the Ridge model to understand feature importance.

In [ ]:
# Using the best alpha (e.g., alpha=1)
best_model = Ridge(alpha=1)
best_model.fit(X_train_np, y_train)

feature_importance = []
features = X_train.columns
coeffs = best_model.coef_

for i in range(len(features)):
    feature_importance.append((features[i], coeffs[i], abs(coeffs[i])))

# Sort by absolute value
feature_importance.sort(key=lambda x: x[2], reverse=True)

print(f"{'Feature':<15} {'Coefficient':>12}")
print("-" * 30)
for feat, coef, _ in feature_importance:
    print(f"{feat:<15} {coef:12.4f}")

### Questions
1. **How does increasing $\alpha$ affect the scores?**
   - Increasing $\alpha$ generally decreases both training and validation $R^2$ as it adds more constraint to the model.
2. **Which value of $\alpha$ is best?**
   - $\alpha=0$ or $\alpha=1$ seems best as they provide the highest validation $R^2$.
3. **Why does a very large $\alpha$ lead to poor performance?**
   - A very large $\alpha$ causes the coefficients to shrink towards zero, leading to underfitting (high bias).
4. **What does the sign of a coefficient indicate?**
   - A positive coefficient means the feature has a positive relationship with `log_fare`, while a negative one means an inverse relationship.
5. **Which features have the largest impact?**
   - Typically, `pclass` and `age` have significant impacts on the fare.

## III. Gradient Descent

Implementing linear regression using iterative optimization.

In [ ]:
# Add bias term
X_train_bias = np.c_[np.ones(X_train_np.shape[0]), X_train_np]
X_val_bias = np.c_[np.ones(X_val_np.shape[0]), X_val_np]

# Initialize weights
n_features = X_train_bias.shape[1]
w = np.zeros(n_features)

# Hyperparameters
learning_rate = 0.01
epochs = 1000
m = len(X_train_bias)
cost_history = []

# Gradient Descent Loop
for epoch in range(epochs):
    y_pred = X_train_bias @ w
    errors = y_pred - y_train
    gradient = (1/m) * (X_train_bias.T @ errors)
    w = w - learning_rate * gradient
    
    cost = np.mean(errors**2)
    cost_history.append(cost)
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Cost = {cost:.4f}")

# Final Evaluation
y_train_pred_gd = X_train_bias @ w
r2_train_gd = r2_score(y_train, y_train_pred_gd)
print(f"Final Training R2 (GD): {r2_train_gd:.4f}")

### Visualizing Convergence

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(cost_history)
plt.xlabel('Epoch')
plt.ylabel('Cost (MSE)')
plt.title('Gradient Descent Convergence')
plt.grid(True)
plt.show()

### Experimenting with Learning Rates

We try different learning rates to see their effect on convergence.

In [ ]:
learning_rates = [0.0001, 0.001, 0.01, 0.1, 0.5]
plt.figure(figsize=(12, 6))

for lr in learning_rates:
    w_lr = np.zeros(n_features)
    costs = []
    for epoch in range(500): # Fewer epochs for comparison
        y_pred = X_train_bias @ w_lr
        errors = y_pred - y_train
        gradient = (1/m) * (X_train_bias.T @ errors)
        w_lr = w_lr - lr * gradient
        costs.append(np.mean(errors**2))
    plt.plot(costs, label=f'lr={lr}')

plt.xlabel('Epoch')
plt.ylabel('Cost (MSE)')
plt.title('Effect of Learning Rate on Convergence')
plt.legend()
plt.grid(True)
plt.show()

### Questions
1. **How does the cost decrease indicate that GD is working?**
   - The cost decreases rapidly at first and then levels off, showing that the model is converging to a minimum.
2. **What might happen if the learning rate is too high or too low?**
   - Too high: The cost might oscillate or diverge.
   - Too low: Convergence will be very slow.
3. **What does the shape of the cost curve tell us?**
   - The smooth exponential-like decay indicates a well-chosen learning rate and successful convergence.
4. **Try different values of learning rate. Explain the effect.**
   - Small rates (0.0001) converge very slowly.
   - Moderate rates (0.01, 0.1) converge efficiently.
   - Very high rates might cause the cost to explode or oscillate.